<a href="https://colab.research.google.com/github/bimaadiwijaya8/-ML_05_Bima-Adiwijaya/blob/main/JS06/JS06-Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [2]:
# Memuat dataset voice.csv (pastikan file voice.csv ada di folder yang sama)
df = pd.read_csv('voice.csv')

# Memisahkan fitur dan target (label 'male' / 'female')
X = df.drop('label', axis=1)
y = df['label']

print("Ukuran dataset:", df.shape)
df.head()

Ukuran dataset: (3168, 21)


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [3]:
ratios = [(0.7, 0.3), (0.8, 0.2)]
kernels = ['linear', 'poly', 'rbf']

results = []

for train_size, test_size in ratios:
    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    # Standarisasi Fitur
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Melatih SVM untuk setiap kernel
    for kernel in kernels:
        svm = SVC(kernel=kernel, random_state=42)
        svm.fit(X_train_scaled, y_train)

        # Prediksi dan Akurasi
        y_pred = svm.predict(X_test_scaled)
        acc = accuracy_score(y_test, y_pred)

        results.append({
            'Rasio (Train:Test)': f"{int(train_size*100)}:{int(test_size*100)}",
            'Kernel': kernel.capitalize(),
            'Akurasi (%)': round(acc * 100, 2)
        })

In [4]:
df_results = pd.DataFrame(results)
df_results

,Rasio (Train:Test),Kernel,Akurasi (%)
0,70:30,Linear,97.90
1,70:30,Poly,96.00
2,70:30,Rbf,98.32
3,80:20,Linear,97.48
4,80:20,Poly,95.58
5,80:20,Rbf,98.26


# 2.

In [5]:
import os
import cv2
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

In [9]:
!unzip images.zip

Archive:  images.zip
   creating: images/
  inflating: __MACOSX/._images       
   creating: images/test/
  inflating: __MACOSX/images/._test  
   creating: images/training/
  inflating: __MACOSX/images/._training  
   creating: images/test/day/
  inflating: __MACOSX/images/test/._day  
   creating: images/test/night/
  inflating: __MACOSX/images/test/._night  
   creating: images/training/day/
  inflating: __MACOSX/images/training/._day  
   creating: images/training/night/
  inflating: __MACOSX/images/training/._night  
  inflating: images/test/day/20151102_122322.jpg  
  inflating: __MACOSX/images/test/day/._20151102_122322.jpg  
  inflating: images/test/day/20151101_235039.jpg  
  inflating: __MACOSX/images/test/day/._20151101_235039.jpg  
  inflating: images/test/day/20151102_114041.jpg  
  inflating: __MACOSX/images/test/day/._20151102_114041.jpg  
  inflating: images/test/day/20151102_074010.jpg  
  inflating: __MACOSX/images/test/day/._20151102_074010.jpg  
  inflating: images/

In [10]:
def extract_color_histogram(image_path, bins=(8, 8, 8)):
    image = cv2.imread(image_path)
    if image is None:
        return None

    # Konversi BGR ke HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Hitung histogram 3D untuk saluran H, S, V dan normalisasi
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)

    return hist.flatten()

def load_data_from_dir(base_dir):
    features = []
    labels = []

    # Membaca kelas/folder ('day' dan 'night')
    for category in ['day', 'night']:
        folder_path = os.path.join(base_dir, category)
        if not os.path.exists(folder_path):
            continue

        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)
            hist = extract_color_histogram(img_path)
            if hist is not None:
                features.append(hist)
                labels.append(category)

    return np.array(features), np.array(labels)

# Memuat data latih dan data uji dari folder direktori spesifik
X_train_img, y_train_img = load_data_from_dir('images/training')
X_test_img, y_test_img = load_data_from_dir('images/test')

print("Jumlah Data Latih (Training) :", len(X_train_img))
print("Jumlah Data Uji (Test)       :", len(X_test_img))

Jumlah Data Latih (Training) : 240
Jumlah Data Uji (Test)       : 160


In [11]:
scaler_img = StandardScaler()

# Scaler di-fit HANYA pada data training, lalu di-transform ke training dan test
X_train_img_scaled = scaler_img.fit_transform(X_train_img)
X_test_img_scaled = scaler_img.transform(X_test_img)

print("Bentuk Fitur Data Latih :", X_train_img_scaled.shape)
print("Bentuk Fitur Data Uji   :", X_test_img_scaled.shape)

Bentuk Fitur Data Latih : (240, 512)
Bentuk Fitur Data Uji   : (160, 512)


In [14]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

grid_search = GridSearchCV(
    estimator=SVC(kernel='rbf', random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train_img_scaled, y_train_img)

print("Hyperparameter RBF Terbaik :", grid_search.best_params_)
print("Akurasi Cross-Validation   :", f"{grid_search.best_score_ * 100:.2f}%")

Hyperparameter RBF Terbaik : {'C': 1, 'gamma': 'scale'}
Akurasi Cross-Validation   : 99.58%


In [15]:
best_model = grid_search.best_estimator_
y_pred_img = best_model.predict(X_test_img_scaled)

acc_img = accuracy_score(y_test_img, y_pred_img)

print("==========================================")
print(f"Akurasi Pengujian Data Test: {acc_img * 100:.2f}%")
print("==========================================\n")
print("Laporan Klasifikasi:\n", classification_report(y_test_img, y_pred_img))

Akurasi Pengujian Data Test: 88.12%

Laporan Klasifikasi:
               precision    recall  f1-score   support

         day       0.97      0.79      0.87        80
       night       0.82      0.97      0.89        80

    accuracy                           0.88       160
   macro avg       0.90      0.88      0.88       160
weighted avg       0.90      0.88      0.88       160

